# Starlink VHF两个事件：时间计算与多尺度瀑布图

从两个已通过LoRa payload CRC验证的解调结果中读取packet时间，随后对原始480 MS/s实数电压进行宽通带DDC，绘制三种尺度：

1. **File context**：完整约8.53 s文件，说明事件在整段记录中的位置与周围频谱背景；
2. **Full packet**：完整LoRa packet及前后背景，标记preamble、PHY data和packet stop；
3. **Chirp detail**：preamble开始后的150 ms，显示逐个LoRa chirp及频率回绕。


第一次运行DDC耗时较长；缓存存在时会直接读取。两个事件采用相同的时频参数和色标，避免各自自动拉伸造成强弱误判。

# 1 设置

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import json
import re

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, FormatStrFormatter
import numpy as np
import pandas as pd
from scipy import signal

try:
    from lfdemod.starlink_vhf import (
        StarlinkVHFDDCConfig,
        downconvert_starlink_vhf_voltage,
    )
except ImportError as error:
    raise ImportError(
        '请先安装LFdemod及其Starlink VHF可选依赖。'
    ) from error

PROJECT_DIR = Path('')
RAW_DATA_DIR = PROJECT_DIR / 'raw_data'
DEMOD_DIR = PROJECT_DIR / '2.2_demodulation_result'
RESULT_DIR = PROJECT_DIR / 'waterfall_results'
CACHE_DIR = RESULT_DIR / 'iq_cache'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)



RAW_SAMPLE_RATE_HZ = 480e6
RAW_DTYPE = np.dtype('<i2')
RF_CENTER_HZ = 137.055e6

EVENT_SUMMARIES = {
    'Event 1 (20250415)': DEMOD_DIR / 'file0' / 'summary.json',
    'Event 2 (20250415)': DEMOD_DIR / 'file12' / 'summary.json',
}

# 所有panel共享同一色标；单位是相对各频率局部中位噪声的dB。
COLOR_LIMITS_DB = (-5.0, 35.0)
# 图1中(a)、(b)子图的频道中心图例字号。
OVERVIEW_LEGEND_FONT_SIZE = 8
SAVE_IQ_CACHE = True
OVERWRITE_IQ_CACHE = False

print('结果目录:', RESULT_DIR)
print('时间基准: 每个.dat文件的mtime作为文件结束时刻')

# 2. 从解调结果计算事件时间

- **接收时间**：把每个`.dat`的最后修改时间（mtime）作为该文件结束时刻，倒退该文件真实样本时长得到开始时刻，再加上packet在文件内的位置；


In [ ]:
def file_number(path):
    """Return the integer suffix in a name such as ...-12.dat."""
    match = re.search(r'-(\d+)\.dat$', Path(path).name)
    if match is None:
        raise ValueError(f'无法从文件名提取编号: {path}')
    return int(match.group(1))


def file_duration_seconds(path):
    """Return raw real-int16 file duration in seconds."""
    path = Path(path)
    sample_count = path.stat().st_size // RAW_DTYPE.itemsize
    return sample_count / RAW_SAMPLE_RATE_HZ


def file_end_utc(path):
    """Return filesystem modification time as a timezone-aware UTC datetime."""
    path = Path(path)
    return datetime.fromtimestamp(path.stat().st_mtime_ns / 1e9, tz=timezone.utc)


def load_event_metadata(label, summary_path):
    """Return one decoded event and its calculated reception UTC fields."""
    summary_path = Path(summary_path)
    summary = json.loads(summary_path.read_text())
    if summary['packet_count'] != 1:
        raise ValueError(f'{label}预期恰好一个packet，实际为{summary["packet_count"]}')
    packet = summary['packets'][0]
    raw_path = RAW_DATA_DIR / Path(summary['input_path']).name
    index = file_number(raw_path)
    duration_s = file_duration_seconds(raw_path)
    end_utc = file_end_utc(raw_path)
    file_start_utc = end_utc - timedelta(seconds=duration_s)
    preamble_s = float(packet['preamble_start_in_file_seconds'])
    phy_s = float(packet['phy_data_start_in_file_seconds'])
    stop_s = float(packet['packet_stop_in_file_seconds'])
    return {
        'label': label,
        'summary_path': summary_path,
        'raw_path': raw_path,
        'file_index': index,
        'file_duration_s': duration_s,
        'file_start_utc': file_start_utc,
        'file_end_utc': end_utc,
        'preamble_start_s': preamble_s,
        'phy_data_start_s': phy_s,
        'packet_stop_s': stop_s,
        'packet_duration_s': stop_s - preamble_s,
        'preamble_start_utc': file_start_utc + timedelta(seconds=preamble_s),
        'packet_stop_utc': file_start_utc + timedelta(seconds=stop_s),
        'carrier_offset_hz': float(packet['carrier_offset_hz']),
        'observed_center_hz': float(packet['observed_center_hz']),
        'crc_valid': bool(packet['crc_valid']),
        'symbol_count': int(packet['symbol_count']),
    }


events = [
    load_event_metadata(label, path)
    for label, path in EVENT_SUMMARIES.items()
]

In [ ]:
metadata_table = pd.DataFrame([
    {
        'event': event['label'],
        'file': event['raw_path'].name,
        'file_index': event['file_index'],
        'file_duration_s': event['file_duration_s'],
        'file_start_utc': event['file_start_utc'].isoformat(),
        'file_end_utc_mtime': event['file_end_utc'].isoformat(),
        'preamble_start_in_file_s': event['preamble_start_s'],
        'phy_data_start_in_file_s': event['phy_data_start_s'],
        'packet_stop_in_file_s': event['packet_stop_s'],
        'packet_duration_s': event['packet_duration_s'],
        'preamble_start_utc': event['preamble_start_utc'].isoformat(),
        'packet_stop_utc': event['packet_stop_utc'].isoformat(),
        'carrier_offset_hz': event['carrier_offset_hz'],
        'observed_center_hz': event['observed_center_hz'],
        'symbol_count': event['symbol_count'],
        'crc_valid': event['crc_valid'],
    }
    for event in events
])
metadata_path = RESULT_DIR / 'event_waterfall_metadata.csv'
metadata_table.to_csv(metadata_path, index=False)
display(metadata_table.T)
print('Saved:', metadata_path)

# 3. 宽通带DDC并缓存整文件IQ

解码器默认DDC只需覆盖约41.7 kHz LoRa信号。为了让整文件背景图显示更宽的邻近频段，这里仍调用LFdemod相同的通用DDC，但将输出设为480 kS/s、有效通带设为约±150 kHz。该设置只服务于绘图，不改变解码结果。

In [ ]:
WATERFALL_DDC = StarlinkVHFDDCConfig(
    fs_in=RAW_SAMPLE_RATE_HZ,
    fs_mid=2.4e6,
    fs_out=480e3,
    first_stage_passband_hz=300e3,
    first_stage_stopband_hz=1.0e6,
    channel_passband_hz=150e3,
    channel_stopband_hz=220e3,
    passband_ripple_db=0.5,
    stopband_attenuation_db=70.0,
    chunk_samples=5_000_000,
)
WATERFALL_DDC.validate()


def downconvert_event(event, config=WATERFALL_DDC):
    """Return whole-file complex IQ, reading/writing a reusable .npy cache."""
    cache_name = (
        f'{event["raw_path"].stem}_'
        f'{RF_CENTER_HZ / 1e6:.3f}MHz_{config.fs_out / 1e3:.0f}ksps.npy'
    )
    cache_path = CACHE_DIR / cache_name
    if cache_path.exists() and not OVERWRITE_IQ_CACHE:
        print(event['label'], '读取缓存:', cache_path)
        return np.load(cache_path, mmap_mode='r'), cache_path

    print(event['label'], 'DDC处理中，首次运行会较慢:', event['raw_path'])
    raw = np.memmap(event['raw_path'], dtype=RAW_DTYPE, mode='r')
    iq, _ = downconvert_starlink_vhf_voltage(raw, RF_CENTER_HZ, config)
    if SAVE_IQ_CACHE:
        np.save(cache_path, iq)
        print('Saved cache:', cache_path)
    return iq, cache_path


iq_by_event = {}
for event in events:
    iq, cache_path = downconvert_event(event)
    iq_by_event[event['label']] = iq
    event['iq_cache_path'] = cache_path
    print(event['label'], 'IQ samples:', len(iq), 'duration:', len(iq) / WATERFALL_DDC.fs_out, 's')

# 4. 计算相对局部噪声的瀑布图

每个频率bin都减去packet之外背景数据的中位功率，因此色标表示相对于该频率局部背景的dB增量。这会抑制接收链频响纹理，同时保留突发信号。

In [ ]:
def select_noise_reference(iq, sample_rate_hz, packet_start_s, packet_stop_s, guard_s=0.30, max_seconds=2.0):
    """Return up to max_seconds of IQ outside a guarded packet interval."""
    n = len(iq)
    left_stop = max(0, int(round((packet_start_s - guard_s) * sample_rate_hz)))
    right_start = min(n, int(round((packet_stop_s + guard_s) * sample_rate_hz)))
    target = int(round(max_seconds * sample_rate_hz))
    half = target // 2
    left = np.asarray(iq[max(0, left_stop - half):left_stop])
    right = np.asarray(iq[right_start:min(n, right_start + half)])
    noise = np.concatenate([left, right])
    if len(noise) < 1024:
        raise ValueError('packet之外的背景IQ不足，无法估计噪声基线。')
    return noise


def _spectrogram_power(iq, sample_rate_hz, nperseg, hop):
    """Return FFT-shifted frequency, time and PSD arrays."""
    frequencies, times, power = signal.spectrogram(
        np.asarray(iq),
        fs=sample_rate_hz,
        window='hann',
        nperseg=nperseg,
        noverlap=nperseg - hop,
        detrend=False,
        return_onesided=False,
        scaling='density',
        mode='psd',
    )
    return (
        np.fft.fftshift(frequencies),
        times,
        np.fft.fftshift(power, axes=0),
    )


def relative_spectrogram(iq, noise_iq, sample_rate_hz, start_s, stop_s, nperseg, hop):
    """Return a time crop as dB above the per-frequency median noise PSD."""
    start_s = max(0.0, float(start_s))
    stop_s = min(len(iq) / sample_rate_hz, float(stop_s))
    first = int(round(start_s * sample_rate_hz))
    last = int(round(stop_s * sample_rate_hz))
    frequencies, times, power = _spectrogram_power(
        iq[first:last], sample_rate_hz, nperseg, hop
    )
    noise_frequencies, _, noise_power = _spectrogram_power(
        noise_iq, sample_rate_hz, nperseg, hop
    )
    if not np.allclose(frequencies, noise_frequencies):
        raise RuntimeError('信号与噪声spectrogram频率轴不一致。')
    tiny = np.finfo(np.float64).tiny
    baseline_db = np.median(10.0 * np.log10(np.maximum(noise_power, tiny)), axis=1)
    relative_db = 10.0 * np.log10(np.maximum(power, tiny)) - baseline_db[:, None]
    return {
        'frequency_offset_hz': frequencies,
        'time_in_file_s': times + start_s,
        'relative_db': relative_db,
        'nperseg': nperseg,
        'hop': hop,
    }


def compute_event_spectrograms(event, iq, sample_rate_hz):
    """Return context, complete-packet and preamble-detail spectrograms."""
    noise_iq = select_noise_reference(
        iq, sample_rate_hz, event['preamble_start_s'], event['packet_stop_s']
    )
    packet_start = max(0.0, event['preamble_start_s'] - 0.10)
    packet_stop = min(event['file_duration_s'], event['packet_stop_s'] + 0.10)
    detail_start = event['preamble_start_s']
    detail_stop = min(event['packet_stop_s'], detail_start + 0.150)
    return {
        'context': relative_spectrogram(
            iq, noise_iq, sample_rate_hz, 0.0, event['file_duration_s'],
            nperseg=1024, hop=512,
        ),
        'packet': relative_spectrogram(
            iq, noise_iq, sample_rate_hz, packet_start, packet_stop,
            nperseg=1024, hop=128,
        ),
        'detail': relative_spectrogram(
            iq, noise_iq, sample_rate_hz, detail_start, detail_stop,
            nperseg=1024, hop=64,
        ),
    }


spectrograms = {}
for event in events:
    spectrograms[event['label']] = compute_event_spectrograms(
        event, iq_by_event[event['label']], WATERFALL_DDC.fs_out
    )
    for scale, spec in spectrograms[event['label']].items():
        print(event['label'], scale, 'shape=', spec['relative_db'].shape,
              '99.9%=', f'{np.percentile(spec["relative_db"], 99.9):.1f} dB')

# 5. 绘制论文候选图

三种尺度分别单独成图；每张图均将事件一和事件二左右并排，便于直接比较。

In [ ]:
def _set_utc_axis(ax):
    """Apply the mtime-derived UTC formatter to a whole-file axis."""
    ax.xaxis.set_major_locator(mdates.SecondLocator(interval=2, tz=timezone.utc))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S', tz=timezone.utc))
    ax.xaxis.set_minor_locator(AutoMinorLocator(4))


def draw_waterfall(ax, event, spec, scale, frequency_half_span_khz, panel_label):
    """Draw one waterfall with absolute UTC for context and relative zoom time."""
    time_s = spec['time_in_file_s']
    if scale == 'context':
        file_start_number = mdates.date2num(event['file_start_utc'])
        x = file_start_number + time_s / 86400.0
        xlabel = 'Time (UTC)'
    elif scale == 'packet':
        x = time_s - event['preamble_start_s']
        xlabel = 'Time from preamble start (s)'
    elif scale == 'detail':
        x = (time_s - event['preamble_start_s']) * 1e3
        xlabel = 'Time from preamble start (ms)'
    else:
        raise ValueError(scale)

    if scale == 'context':
        frequency_axis = (RF_CENTER_HZ + spec['frequency_offset_hz']) / 1e6
        frequency_limits = (
            (RF_CENTER_HZ - frequency_half_span_khz * 1e3) / 1e6,
            (RF_CENTER_HZ + frequency_half_span_khz * 1e3) / 1e6,
        )
        ylabel = 'Frequency (MHz)'
    else:
        frequency_axis = spec['frequency_offset_hz'] / 1e3
        frequency_limits = (-frequency_half_span_khz, frequency_half_span_khz)
        ylabel = f'Frequency offset (kHz)'

    mesh = ax.pcolormesh(
        x, frequency_axis, spec['relative_db'],
        shading='auto', cmap='viridis', rasterized=True,
        vmin=COLOR_LIMITS_DB[0], vmax=COLOR_LIMITS_DB[1],
    )
    ax.set_ylim(*frequency_limits)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if scale == 'context':
        _set_utc_axis(ax)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
        ax.axhline(RF_CENTER_HZ / 1e6, color='white', lw=0.9,
                   ls='--', alpha=0.85,
                   label=f'Starlink VHF TT&C center ({RF_CENTER_HZ / 1e6:.3f} MHz)')
    ax.text(0.012, 0.95, panel_label, transform=ax.transAxes,
            va='top', ha='left', color='white', fontsize=12, fontweight='bold')

    if scale == 'packet':
        ax.axvline(0.0, color='white', lw=1.0, ls='--', label='Preamble start')
        ax.axvline(event['phy_data_start_s'] - event['preamble_start_s'],
                   color='cyan', lw=1.0, ls=':', label='PHY data start')
        ax.axvline(event['packet_stop_s'] - event['preamble_start_s'],
                   color='white', lw=1.0, ls='--', label='Packet stop')
    return mesh


def save_figure(fig, stem):
    """Save one figure as high-resolution PNG and vector PDF."""
    png_path = RESULT_DIR / f'{stem}.png'
    pdf_path = RESULT_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    print('Saved:', png_path)
    print('Saved:', pdf_path)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True, sharey=True)
last_mesh = None
for column, event in enumerate(events):
    last_mesh = draw_waterfall(
        axes[column], event, spectrograms[event['label']]['context'],
        'context', 140.0, f'({chr(ord("a") + column)})',
    )
    axes[column].set_title(
        event['label'], fontsize=11
    )
axes[1].set_ylabel('')
for ax in axes:
    ax.legend(
        loc='lower right', fontsize=OVERVIEW_LEGEND_FONT_SIZE,
        framealpha=0.75,
    )
colorbar = fig.colorbar(last_mesh, ax=axes, location='right', shrink=0.92, pad=0.015)
colorbar.set_label('Power above local median noise (dB)')
save_figure(fig, 'starlink_two_events_overview')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True, sharey=True)
last_mesh = None
for column, event in enumerate(events):
    last_mesh = draw_waterfall(
        axes[column], event, spectrograms[event['label']]['packet'],
        'packet', 45.0, f'({chr(ord("a") + column)})',
    )
    axes[column].set_title(
        event['label'], fontsize=11
    )
axes[1].set_ylabel('')
for ax in axes:
    ax.legend(loc='upper right', fontsize=8, framealpha=0.75)
colorbar = fig.colorbar(last_mesh, ax=axes, location='right', shrink=0.92, pad=0.015)
colorbar.set_label('Power above local median noise (dB)')
save_figure(fig, 'starlink_two_events_full_packet')
plt.show()

In [ ]:

symbol_duration_ms = (2**8) / (125_000.0 / 3.0) * 1e3
structure_boundaries_ms = {
    'Preamble / sync': 15.0 * symbol_duration_ms,
    'Sync / SFD': (15.0 + 2.0) * symbol_duration_ms,
    'SFD / PHY data': (15.0 + 2.0 + 2.25) * symbol_duration_ms,
}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True, sharey=True)
last_mesh = None
for column, event in enumerate(events):
    last_mesh = draw_waterfall(
        axes[column], event, spectrograms[event['label']]['detail'],
        'detail', 32.0, f'({chr(ord("a") + column)})',
    )
    axes[column].set_title(event['label'], fontsize=11)
    for boundary_ms in structure_boundaries_ms.values():
        axes[column].axvline(
            boundary_ms, color='white', lw=0.8, ls='--',
            alpha=0.85, zorder=3,
        )
axes[1].set_ylabel('')
colorbar = fig.colorbar(last_mesh, ax=axes, location='right', shrink=0.92, pad=0.015)
colorbar.set_label('Power above local median noise (dB)')
save_figure(fig, 'starlink_two_events_sync_detail_boundaries')
plt.show()

for name, boundary_ms in structure_boundaries_ms.items():
    print(f'{name}: {boundary_ms:.3f} ms')

# 6. 论文组合图：三种尺度合并为一张图


三行都使用相同的1024-point Hann窗和统一的`-5–35 dB`色标，但hop分别为512、128和64 samples。因此，这是显示范围逐级缩小且时间列更密的三种视图，而不是三种不同的频率bin间隔。

In [ ]:
# 三行两列论文组合图：左列Event 1，右列Event 2。
COMBINED_FIGSIZE = (15.0, 13.2)
COMBINED_PANEL_LABEL_SIZE = 12
COMBINED_TITLE_SIZE = 12
COMBINED_LEGEND_SIZE = 7.5

fig, axes = plt.subplots(
    3, 2, figsize=COMBINED_FIGSIZE, constrained_layout=True,
    gridspec_kw={'height_ratios': [1.0, 1.0, 1.0]},
)

row_settings = [
    ('context', 140.0),
    ('packet', 45.0),
    ('detail', 32.0),
]
panel_index = 0
last_mesh = None

for row, (scale, frequency_half_span_khz) in enumerate(row_settings):
    for column, event in enumerate(events):
        ax = axes[row, column]
        panel_label = f'({chr(ord("a") + panel_index)})'
        last_mesh = draw_waterfall(
            ax, event, spectrograms[event['label']][scale],
            scale, frequency_half_span_khz, panel_label,
        )
        # draw_waterfall中的panel字号在此统一，便于后续调整。
        ax.texts[-1].set_fontsize(COMBINED_PANEL_LABEL_SIZE)
        if row == 0:
            ax.set_title(event['label'], fontsize=COMBINED_TITLE_SIZE)
        if column == 1:
            ax.set_ylabel('')

        if scale == 'context':
            ax.legend(
                loc='lower right', fontsize=COMBINED_LEGEND_SIZE,
                framealpha=0.75,
            )
        elif scale == 'packet':
            ax.legend(
                loc='upper right', fontsize=COMBINED_LEGEND_SIZE,
                framealpha=0.75,
            )
        else:
            for boundary_ms in structure_boundaries_ms.values():
                ax.axvline(
                    boundary_ms, color='white', lw=0.8, ls='--',
                    alpha=0.85, zorder=3,
                )
        panel_index += 1

colorbar = fig.colorbar(
    last_mesh, ax=axes, location='right', shrink=0.98, pad=0.015,
)
colorbar.set_label('Power above local median noise (dB)')
save_figure(fig, 'starlink_two_events_three_scale_combined')
plt.show()